# Project: Clinical Note Structured Extractor

## Objective

Transform unstructured clinical notes into structured, validated JSON output suitable for:

- Analytics
- Risk stratification
- Dataset generation
- Integration with FHIR

## ML model pipelines

- **System Architecture (Conceptual)**

    Pipeline:

    -- Clinical Note
    -- Prompt Template
    -- LLM
    -- Structured Output Parser (Pydantic)
    -- JSON
    -- (Optional) FHIR Mapping




- **Define the Clinical Schema (Scientific Rigor)**

    Before coding, define a medically coherent schema. Example minimal schema:

    - Chief complaint
    - Symptoms
    - Duration
    - Vital signs
    - Risk level
    - Differential diagnosis
    - Red flags
    - This enforces semantic discipline.

Patient is a 65-year-old male presenting with chest pain for 3 days.
Pain radiates to left arm. Reports shortness of breath.
BP 150/95 mmHg. HR 102 bpm.
History of hypertension and diabetes.

## Define Structured Schema

In [ ]:
# from pydantic import BaseModel, Field
# from typing import List, Optional

# class ClinicalExtraction(BaseModel):
#     age: Optional[int] = Field(description="Patient age")
#     sex: Optional[str] = Field(description="Biological sex")
#     chief_complaint: Optional[str]
#     symptoms: List[str]
#     duration_days: Optional[int]
#     vital_signs: Optional[dict]
#     risk_level: Optional[str]

In [9]:
from pydantic import BaseModel, Field
from typing import List, Optional, Dict

class ClinicalExtraction(BaseModel):
    age: Optional[int] = None
    sex: Optional[str] = None
    chief_complaint: Optional[str] = None
    symptoms: Optional[List[str]] = None
    duration_days: Optional[int] = None
    vital_signs: Optional[Dict[str, str]] = None
    risk_level: Optional[str] = None

In [11]:
#from langchain.chat_models import ChatOpenAI
#from langchain.prompts import ChatPromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_ollama import ChatOllama

# prompt = ChatPromptTemplate.from_template("""
# You are a clinical information extraction system.

# Extract structured clinical information from the note.

# {format_instructions}

# Clinical note:
# {note}
# """)

prompt = ChatPromptTemplate.from_template("""
You are a clinical information extraction system.

Extract structured clinical information from the note below.

IMPORTANT:
- Return ONLY a valid JSON object.
- Do NOT return the schema.
- Do NOT explain anything.
- Do NOT include markdown.
- The output must match the schema exactly.

{format_instructions}

Clinical note:
{note}
""")

parser = PydanticOutputParser(pydantic_object=ClinicalExtraction)

llm = ChatOllama(model='tinyllama', temperature=0, format="json")

#llm = ChatOpenAI(
#    temperature=0,
#    model="gpt-4o-mini"
#)


chain = prompt | llm | parser

In [16]:
note = """
Patient is a 65-year-old male presenting with chest pain for 3 days.
Pain radiates to left arm. Reports shortness of breath.
BP 150/95 mmHg. HR 102 bpm.
History of hypertension and diabetes.
"""

result = chain.invoke({
    "note": note,
    "format_instructions": parser.get_format_instructions()
})

print(result.model_dump_json())

{"age":null,"sex":null,"chief_complaint":null,"symptoms":null,"duration_days":null,"vital_signs":null,"risk_level":null}


In [17]:
structured_llm = llm.with_structured_output(ClinicalExtraction)
result = structured_llm.invoke(note)

print(result)

age=None sex=None chief_complaint=None symptoms=['chest pain', 'shortness of breath', 'reported shortness of breath', 'BP 150/95 mmHg', 'HR 102 bpm'] duration_days=3 vital_signs=None risk_level='high risk of hypertenseion and diabetes mellitus'
